In [2]:
import numpy as np
import matplotlib.pyplot as plt
import random
import time
import seaborn as sns
import colorcet as cc
import collections
import sys
import multiprocessing

from IPython.display import clear_output
from matplotlib.path import Path
from astropy import units as u
from astropy.coordinates import SkyCoord,Angle
from astropy.table import Table,vstack
from itertools import chain
from sklearn.cluster import DBSCAN,AgglomerativeClustering
from sklearn.neighbors import KernelDensity
from functools import partial

#below for netflow
from __future__ import print_function
import ets_fiber_assigner.netflow as nf
from ics.cobraOps.Bench import Bench
from ics.cobraOps.TargetGroup import TargetGroup
from ics.cobraOps.CobrasCalibrationProduct import CobrasCalibrationProduct
from ics.cobraOps.CollisionSimulator import CollisionSimulator
from ics.cobraOps.cobraConstants import NULL_TARGET_POSITION, NULL_TARGET_ID
from ics.cobraOps import plotUtils
from collections import defaultdict

import mercury as mr # for widgets

import warnings
warnings.filterwarnings('ignore')

In [2]:
app = mr.App(title="PPP online tool", description="webApp_v1",show_code=False)

mercury.App

In [45]:
def readUserSample(file,Print):
    '''Read user sample from the file: 
    in format of 'obj_id' 'ra' 'dec' 'equinox' 'priority' 'effective_exp' 'resolution'

    Parameters
    ==========
    direc : string
        the name of directory containing sample lists

    Returns
    =======
    user sample (all), user sample (low-resolution mode), user sample (medium-resolution mode)
    '''
    time_start=time.time()
    
    # quit if no sample uploaded
    if file.filepath is None:
        sys.exit("No input sample") 
    
    Formlist=['csv','fits','ecsv']
    Form=[tt for tt in Formlist if tt in file.filename][0]
    uSamp=Table.read(file.filepath,format=Form) 
    
    # quit if the required information missed in the uploaded sample 
    info_need=np.array(['obj_id', 'ra', 'dec', 'equinox', 'priority', 'exptime', 'resolution'])
    info_miss=info_need[np.in1d(info_need,uSamp.colnames)==False]
    if len(info_miss)!=0:
        sys.exit('The following information is not provided in the uploaded file: '+\
                 ', '.join(info_miss))    
        
    #change ra&dec in the input sample into degree if ra(dec) in hms(dms)
    for i in range(len(uSamp['ra'])):
        if np.issubdtype(type(uSamp['ra'][i]), np.str_) and any([x in uSamp['ra'][i] for x in [':','h']]):
            #hms --> degree
            uSamp['ra'][i]='{:.5f}'.format(Angle(uSamp['ra'][i],unit=u.hour).degree)
        if np.issubdtype(type(uSamp['dec'][i]), np.str_):
            #dms --> degree
            uSamp['dec'][i]='{:.5f}'.format(Angle(uSamp['dec'][i],unit=u.degree).degree)
    uSamp['ra']=uSamp['ra'].astype(float)
    uSamp['dec']=uSamp['dec'].astype(float)
    
    #all obj_id in str format
    uSamp['obj_id']=uSamp['obj_id'].astype(str)
    
    #all exptime needs to be multiple of 900s, otherwise netflow can not read in the info
    #add additional column to store this info
    exptime_ppp=np.ceil(uSamp['exptime']/900)*900
    uSamp.add_column(exptime_ppp,name='exptime_PPP')
    
    #print warning if sample contains objects with dec<-40 (not observable with Subaru)
    if any(uSamp['dec']<-40):
        sys.exit('WARNING: objects with dec<-40 are not observable, please remove them.')
    
    # separete the sample by 'resolution' (L/M)
    allSamp=uSamp.group_by('resolution')
    allSamp_L=allSamp[allSamp['resolution']=='L']
    allSamp_M=allSamp[allSamp['resolution']=='M']
    
    #print info of reading in sample is necessary
    if Print:
        print("#########Step 1/4: Read sample")
        print("#Your input sample is ",file.filename)  
        print("#There are {:8d} targets requiring the LOW resolution".format(len(allSamp_L)))
        print("#There are {:8d} targets requiring the MEDIUM resolution".format(len(allSamp_M)))
        print("#########Step 1/4: Read sample DONE! (takes",round(time.time()-time_start,3),"sec)")
        
    return allSamp,allSamp_L,allSamp_M#'''

In [54]:
_ = mr.Note(text="###### Step1. upload file")
file = mr.File(label="Please upload your file here (size limit: 20MB):", max_file_size="20MB")
if file.filepath is None:
    mr.Stop()
s1=readUserSample(file,True)

###### Step1. upload file

mercury.File

In [1]:
def count_N(sample):
    '''calculate local count of targets

    Parameters
    ==========
    sample : table

    Returns
    =======
    sample added with local density (bin_width is 1 deg in ra&dec)
    '''
    #lower limit of dec is -40
    count_bin=[ [ 0 for i in np.arange(0,361,1) ] for j in np.arange(-40,91,1) ]
    for ii in range(len(sample['ra'])):
        m=int(sample['ra'][ii])
        n=int(sample['dec'][ii]+40) #dec>-40
        count_bin[n][m]+=1
    den_local=[count_bin[int(sample['dec'][ii]+40)][int(sample['ra'][ii])] for ii in range(len(sample['ra']))]
    
    if 'local_count' not in sample.colnames:
        sample.add_column(den_local,name='local_count')
    else:
        sample['local_count']=den_local
        
    return sample

def weight(sample,conta,contb,contc): 
    '''calculate weights of targets (larger weights mean more important)

    Parameters
    ==========
    sample : table
    conta,contb,contc: float
        parameters of weighting scheme: conta--> science grade,>0; contb--> remaining time; contc--> local density
        
    Returns
    =======
    sample: table added with weight col
    '''
    weight_t=pow(conta,2.+0.1*(9-sample['priority']))*pow(sample['exptime_PPP']/900.,contb)*pow(sample['local_count'],contc)
    
    if 'weight' not in sample.colnames:
        sample.add_column(weight_t,name='weight')
    else:
        sample['weight']=weight_t
        
    return sample

def target_DBSCAN(sample,sep=1.38,Print=True):
    '''separate pointings/targets into different groups

    Parameters
    ==========
    sample:table
    sep: float
        angular separation set to group, degree
    Print:boolean
        
    Returns
    =======
    list of pointing centers in different group
    '''    
    #haversine uses (dec,ra) in radian; 
    db = DBSCAN(eps=np.radians(sep), min_samples=1, metric='haversine').fit(np.radians([sample['dec'],sample['ra']]).T)
    
    labels = db.labels_
    unique_labels = set(labels)
    n_clusters = len(unique_labels)
    
    if Print:
        print(f"#There are {len(sample):d} targets, they are grouped into {n_clusters:d} clusters.")
           
    tgt_group=[]
    
    for ii in range(n_clusters):
        tgt_t=sample[labels==ii]
        tgt_group.append(tgt_t)

    return tgt_group

def target_collision(sample,sep=2/3600.,Print=True):
    '''check targets collide with each other

    Parameters
    ==========
    sample:table
    sep: float
        angular separation set define collided targets, degree, default=2 arcsec
    Print:boolean
        
    Returns
    =======
    list of pointing centers in different group
    '''    
    #haversine uses (dec,ra) in radian; 
    db = AgglomerativeClustering(distance_threshold=np.radians(sep),n_clusters=None,affinity='haversine',\
                                  linkage='single').fit(np.radians([sample['dec'],sample['ra']]).T)
    
    labels = db.labels_
    unique_labels = set(labels)
    labels_c=[lab for lab in unique_labels if list(labels).count(lab)>1]
    
    if len(labels_c)==0:
        index=np.arange(0,len(sample),1)
        return index
    else:
        if Print:
            print(f"#There are {len(labels_c):d} collision regions.")
            
        index=list(np.where(np.in1d(labels,labels_c)==False)[0])+\
                [np.where(np.in1d(labels,kk)==True)[0][0] for kk in labels_c]
        return sorted(index)


def PFS_FoV(ppc_ra,ppc_dec,PA,sample,mode=None):
    '''pick up targets in the pointing

    Parameters
    ==========
    ppc_ra,ppc_dec,PA : float
        ra,dec,PA of the pointing center
    sample : table
    mode: default=None
        if "KDE_collision", consider collision avoid in KDE part (separation=2 arcsec)
        
        
    Returns
    =======
    list of index of targets, which fall into the pointing, in the input sample
    '''
    if len(sample)>1 and mode=="KDE_collision":
        index=target_collision(sample)
        point=np.vstack((sample[index]['ra'],sample[index]['dec'])).T
    else:
        point=np.vstack((sample['ra'],sample['dec'])).T
    center=SkyCoord(ppc_ra*u.deg,ppc_dec*u.deg)
    
    #PA=0 along y-axis, PA=90 along x-axis, PA=180 along -y-axis...
    hexagon=center.directional_offset_by([30+PA,90+PA,150+PA,210+PA,270+PA,330+PA,30+PA]*u.deg,1.38/2.*u.deg)
    ra_h=hexagon.ra.deg
    dec_h=hexagon.dec.deg
    
    #for pointings around RA~0 or 360, parts of it will move to the opposite side (e.g., [[1,0],[-1,0]] -->[[1,0],[359,0]])
    #correct for it
    ra_h_in=np.where(np.fabs(ra_h-ppc_ra)>180)
    if len(ra_h_in[0])>0:
        if ra_h[ra_h_in[0][0]]>180:ra_h[ra_h_in[0]]-=360
        elif ra_h[ra_h_in[0][0]]<180:ra_h[ra_h_in[0]]+=360
            
    polygon = Path([(ra_h[t],dec_h[t]) for t in range(len(ra_h))])
    index_=np.where(polygon.contains_points(point)==True)[0]
    
    return index_

def PFS_FoV_plot(ppc_ra,ppc_dec,PA,line_color,line_width,line_st):
    '''plot PFS FoV (hexagon)

    Parameters
    ==========
    ppc_ra,ppc_dec,PA : float
        ra,dec,PA of the pointing center
    line_color,line_st: string
        color and style of the plotting
    line_width: float
        width of the edge of the pointing
        
    Returns
    =======
    plot a hexagon at the pointing center with diameter=1.38 deg
    '''
    if type(ppc_ra) not in [list,np.ndarray]:
        #the input should be given as ndarray, but if it is in the format of float or int, change it to list
        ppc_ra,ppc_dec,PA=[ppc_ra],[ppc_dec],[PA]
    
    #PA=0 along y-axis, PA=90 along x-axis, PA=180 along -y-axis...
    for ii in range(len(ppc_ra)):
        ppc_ra_t,ppc_dec_t,pa_t=ppc_ra[ii],ppc_dec[ii],PA[ii]
        center=SkyCoord(ppc_ra_t*u.deg,ppc_dec_t*u.deg)
        hexagon=center.directional_offset_by([30+pa_t,90+pa_t,150+pa_t,210+pa_t,\
                                              270+pa_t,330+pa_t,30+pa_t]*u.deg,1.38/2.*u.deg)
        ra_h=hexagon.ra.deg
        dec_h=hexagon.dec.deg
    
        #for pointings around RA~0 or 360, parts of it will move to the opposite side (e.g., [[1,0],[-1,0]] -->[[1,0],[359,0]])
        #correct for it
        ra_h_in=np.where(np.fabs(ra_h-center.ra.deg)>180)
        if len(ra_h_in[0])>0:
            if ra_h[ra_h_in[0][0]]>180:ra_h[ra_h_in[0]]-=360
            elif ra_h[ra_h_in[0][0]]<180:ra_h[ra_h_in[0]]+=360
            
        plt.plot(ra_h,dec_h,color=line_color,lw=line_width,ls=line_st,alpha=0.5,zorder=5)
    
def KDE_xy(sample,X,Y):
    '''calculate a single KDE

    Parameters
    ==========
    sample: table
    X,Y: grid to calculate KDE
        
    Returns
    =======
    Z: KDE estimate
    '''
    values = np.vstack((np.deg2rad(sample['dec']),np.deg2rad(sample['ra'])))
    kde = KernelDensity(bandwidth=np.deg2rad(1.38/2.), kernel='linear',algorithm='ball_tree',metric='haversine')
    kde.fit(values.T,sample_weight=sample['weight'])
    
    X1=np.deg2rad(X)
    Y1=np.deg2rad(Y)
    positions = np.vstack([Y1.ravel(),X1.ravel()])    
    Z = np.reshape(np.exp(kde.score_samples(positions.T)), Y.shape)
    
    return Z

def KDE(sample,multiProcesing):
    '''define binning and calculate KDE

    Parameters
    ==========
    sample: table
    multiProcesing: boolean
        allow multiprocessing or not 
        (n_thread set to be the maximal threads allowed in the machine)

    Returns
    =======
    ra_bin, dec_bin, significance of KDE over the field, ra of peak in KDE, dec of peak in KDE
    '''
    if len(sample)==1:
        #if only one target, set it as the peak
        return sample['ra'].data[0],sample['dec'].data[0],np.nan,sample['ra'].data[0],sample['dec'].data[0]
    else:
        #determine the binning for the KDE cal. 
        #set a bin width of 0.5 deg in ra&dec if the sample spans over a wide area (>50 degree)
        #give some blank spaces in binning, otherwide KDE will be wrongly calculated
        ra_low=min(min(sample['ra'])*0.9,min(sample['ra'])-1)
        ra_up=max(max(sample['ra'])*1.1,max(sample['ra'])+1)
        dec_up=max(max(sample['dec'])*1.1,max(sample['dec'])+1)
        dec_low=min(min(sample['dec'])*0.9,min(sample['dec'])-1)
        
        if (max(sample['ra'])-min(sample['ra']))/100<0.5 and (max(sample['dec'])-min(sample['dec']))/100<0.5:
            X_, Y_ = np.mgrid[ra_low:ra_up:101j, dec_low:dec_up:101j]
        elif (max(sample['dec'])-min(sample['dec']))/100<0.5:
            X_, Y_ = np.mgrid[0:360:721j, dec_low:dec_up:101j]
        elif (max(sample['ra'])-min(sample['ra']))/100<0.5:
            X_, Y_ = np.mgrid[ra_low:ra_up:101j, -40:90:261j]
        else:
            X_, Y_ = np.mgrid[0:360:721j, -40:90:261j]
        positions1 = np.vstack([ Y_.ravel(),X_.ravel()])
    
        if multiProcesing:
            threads_count = round(multiprocessing.cpu_count()/2)
            thread_n=min(threads_count,round(len(sample)*0.5)) # threads_count=10 in this machine
            
            with multiprocessing.Pool(thread_n) as p:
                dMap_=p.map(partial(KDE_xy, X=X_,Y=Y_), np.array_split(sample, thread_n))
                
            Z=sum(dMap_)
            
        else:
            Z=KDE_xy(sample,X_,Y_)
        
        #calculate significance level of KDE 
        obj_dis_sig_=(Z-np.mean(Z))/np.std(Z)
        peak_pos=np.where(obj_dis_sig_==obj_dis_sig_.max())
        
        peak_y=positions1[0,peak_pos[1][round(len(peak_pos[1])*0.5)]]
        peak_x=sorted(set(positions1[1,:]))[peak_pos[0][round(len(peak_pos[0])*0.5)]]
        
        return X_,Y_,obj_dis_sig_,peak_x,peak_y

def PPP_centers(sample_f,mutiPro,conta,contb,contc,Print):
    '''determine pointing centers

    Parameters
    ==========
    sample_f : table
    mutiPro : boolean
        allow multiprocessing to calculate KDE or not
    conta,contb,contc: float
        parameters of weighting scheme: conta--> science grade,>0; 
                                        contb--> remaining time; 
                                        contc--> local density
        
    Returns
    =======
    sample with list of pointing centers in meta
    '''
    if Print:
        print("#########Step 2/4: Determine pointing centers")
        
    time_start=time.time()
    Nfiber=int(2394-200) #200 for calibrators
    sample_f=count_N(sample_f)
    sample_f=weight(sample_f,conta,contb,contc)
        
    peak=[]
    
    for sample in target_DBSCAN(sample_f,1.38,Print):
        sample_s=sample[sample['exptime_PPP']>0] #targets not finished
        
        while any(sample_s['exptime_PPP']>0):        
            #-------------------------------
            ####peak_xy from KDE peak with weights 
            X_,Y_,obj_dis_sig_,peak_x,peak_y=KDE(sample_s,mutiPro)
        
            #-------------------------------            
            index_=PFS_FoV(peak_x,peak_y,0,sample_s) # all PA set to be 0 for simplicity
                
            if len(index_)>0:
                peak.append([len(peak),peak_x,peak_y,0]) # ppc_id,ppc_ra,ppc_dec,ppc_PA=0
                
            else:
            #add a small random shift so that it will not repeat over a blank position 
                while len(index_)==0:
                    peak_x_t=peak_x+np.random.choice([0.15,-0.15,0],1)[0]
                    peak_y_t=peak_y+np.random.choice([0.15,-0.15,0],1)[0]
                    index_=PFS_FoV(peak_x_t,peak_y_t,0,sample_s)
                peak.append([len(peak),peak_x_t,peak_y_t,0]) # ppc_id,ppc_ra,ppc_dec,ppc_PA=0
        
            #-------------------------------
            if len(index_)>Nfiber:
                index_=random.sample(list(index_), Nfiber)
            sample_s['exptime_PPP'][list(index_)]-=900 #targets in the PPC observed with 900 sec
        
            sample_s=sample_s[sample_s['exptime_PPP']>0] #targets not finished
            sample_s=count_N(sample_s)
            sample_s=weight(sample_s,conta,contb,contc)
            
    sample_f.meta['PPC']=np.array(peak)
    
    if Print:
        print(f"#There are {len(peak):5d} pointings determined.")
        print(f"#########Step 2/4: Determine pointing centers DONE! (takes {round(time.time()-time_start,3)} sec)")
            
    return sample_f

def plot_KDE(sample_f):
    sample_g=target_DBSCAN(sample_f,1.38,True)
    peak=sample_f.meta['PPC']
    for sample in sample_g:
        plt.figure(figsize=(7,5))
        plt.scatter(sample['ra'],sample['dec'],c=sample['priority'],marker='o',cmap='Paired_r',\
                    vmin=0,vmax=9,s=7,zorder=10)
        plt.colorbar(label='User priority')
        PFS_FoV_plot(peak[:,1],peak[:,2],peak[:,3],'orange',0.5,'--')
        plt.xlim(min(sample['ra'])-1,max(sample['ra'])+1)
        plt.ylim(min(sample['dec'])-1,max(sample['dec'])+1)
        plt.xlabel('RA',fontsize=10)
        plt.ylabel('DEC',fontsize=10)
        plt.title
        plt.show()

In [51]:
thread=20
uS,uS_L,uS_M=s1
conta,contb,contc=[4.02,0.22,0.15]

_ = mr.Note(text="###### Step2. PFS pointing center (PPC) determination")
KDErun = mr.Button(label="Start",style='success')

if KDErun.clicked:
    if len(uS_L)>0: #for low-resolution mode   
        uS_L_s2=PPP_centers(uS_L,True,conta,contb,contc,True)

    if len(uS_M)>0: #for medium-resolution mode
        uS_M_s2=PPP_centers(uS_M,True,conta,contb,contc,True)

###### Step2. PFS pointing center (PPC) determination

mercury.Button

In [52]:
PlotTF = mr.Checkbox(value=False, label="Plot the results.")
if PlotTF.value:
    if len(uS_L)>0: #for low-resolution mode   
        plot_KDE(uS_L_s2)
        
    if len(uS_M)>0: #for medium-resolution mode
        plot_KDE(uS_M_s2)

mercury.Checkbox

In [ ]:
def point_DBSCAN(sample,Plot,Print):
    '''separate pointings into different group

    Parameters
    ==========
    sample:table
    Plot, Print:boolean
        
    Returns
    =======
    list of pointing centers in different group
    '''
    ppc_xy=sample.meta['PPC'] 
    
    #haversine uses (dec,ra) in radian; 
    db = DBSCAN(eps=np.radians(1.38), min_samples=1, metric='haversine').fit(np.fliplr(np.radians(ppc_xy[:,[1,2]])))
    
    labels = db.labels_
    unique_labels = set(labels)
    n_clusters = len(unique_labels)
    
    if Print:
        print("#There are {:5d} pointings, they are grouped into {:5d} clusters.".format(len(ppc_xy),n_clusters))
    
    if Plot:
        colors = sns.color_palette(cc.glasbey_warm, n_clusters)
        
    ppc_group=[]
    
    for ii in range(n_clusters):
        ppc_t=ppc_xy[labels==ii]
        ppc_group.append(ppc_t)

        if Plot:
            xy = ppc_t[:,[1,2]]
            for uu in xy:
                PFS_FoV_plot(uu[0], uu[1],0,colors[ii],0.2,'-')
                plt.plot(uu[0],uu[1],'o',mfc=colors[ii],mew=0,ms=5)
        plt.show()
        
    return ppc_group

def sam2netflow(sample):
    '''put targets to the format which can be read by netflow

    Parameters
    ==========
    sample : table
    
    Returns
    =======
    list of targets readable by netflow
    '''
    targetL=[]
    
    int_=0
    for tt in sample:
        id_, ra, dec, tm= (tt['obj_id'], tt['ra'], tt['dec'],tt['exptime_PPP'])
        targetL.append(nf.ScienceTarget(id_, ra, dec, tm, int_, 'sci'))
        int_+=1
        
    #for ii in range(50): #mock Fstars
    #    targetL.append(nf.CalibTarget('Fs_'+str(ii),0,0, "cal"))
        
    #for jj in range(150):#mock skys
    #    targetL.append(nf.CalibTarget('Sky_'+str(jj),0,0,"sky"))
        
    return targetL

def NetflowPreparation(sample):
    '''assign cost to each target

    Parameters
    ==========
    sample : sample
        
    Returns
    =======
    class of targets with costs
    '''
    
    classdict = {}
    
    int_=0
    for ii in sample:
        classdict["sci_P"+str(int_)] = {"nonObservationCost": ii['weight'],
                                        "partialObservationCost": ii['weight']*1.5, "calib": False}
        int_+=1
        
    #classdict["sky"] = {"numRequired": 150,
    #                    "nonObservationCost": max(sample['weight'])*1., "calib": True}
    
    #classdict["cal"] = {"numRequired": 50,
    #                    "nonObservationCost": max(sample['weight'])*1., "calib": True}
    
    return classdict

def cobraMoveCost(dist):
    '''optional: penalize assignments where the cobra has to move far out
    '''
    return 0.1*dist

def netflowRun_single(Tel,sample,Print=True):
    '''run netflow (without iteration)

    Parameters
    ==========
    sample : sample
    Tel: PPC info (id,ra,dec,PA)
        
    Returns
    =======
    solution of Gurobi, PPC list
    '''
    Telra=Tel[:,1]
    Teldec=Tel[:,2]
    Telpa=Tel[:,3]
    
    bench = Bench(layout="full")
    tgt=sam2netflow(sample)
    classdict=NetflowPreparation(sample)
    otime = "2024-05-20T08:00:00Z"
        
    telescopes = []
    
    nvisit=len(Telra)
    for ii in range(nvisit):
        telescopes.append(nf.Telescope(Telra[ii], Teldec[ii], Telpa[ii], otime))
    tpos = [tele.get_fp_positions(tgt) for tele in telescopes]

    # optional: slightly increase the cost for later observations,
    # to observe as early as possible
    vis_cost = [0 for i in range(nvisit)]

    gurobiOptions=dict(seed=0, presolve=1, method=4, degenmoves=0,heuristics=0.8, mipfocus=0, \
                           mipgap=5.0e-2,LogToConsole=0)
        
    # partially observed? no
    alreadyObserved={}
        
    forbiddenPairs = [[] for i in range(nvisit)]

    # compute observation strategy
    prob = nf.buildProblem(bench, tgt, tpos, classdict, 900,
                            vis_cost, cobraMoveCost=cobraMoveCost,
                            collision_distance=2., elbow_collisions=True,
                            gurobi=True, gurobiOptions=gurobiOptions,
                            alreadyObserved=alreadyObserved,forbiddenPairs=forbiddenPairs)

    prob.solve()
    
    res = [{} for _ in range(min(nvisit,len(Telra)))]
    for k1, v1 in prob._vardict.items():
        if k1.startswith("Tv_Cv_"):
            visited = prob.value(v1) > 0
            if visited:
                _, _, tidx, cidx, ivis = k1.split("_")
                res[int(ivis)][int(tidx)] = int(cidx)
                
    return res,telescopes,tgt

def netflowRun_nofibAssign(Tel,sample,Print=True):
    '''run netflow (with iteration)
        if no fiber assignment in some PPCs, shift these PPCs with 0.2 deg
        
    Parameters
    ==========
    sample : sample
    Tel: PPC info (id,ra,dec,PA)
        
    Returns
    =======
    solution of Gurobi, PPC list
    '''
    res,telescope,tgt=netflowRun_single(Tel,sample)
    
    if sum(np.array([len(tt) for tt in res])==0)==0:
        #All PPCs have fiber assignment
        return res,telescope,tgt
    
    else:
        #if there are PPCs with no fiber assignment
        index=np.where(np.array([len(tt) for tt in res])==0)[0]
    
        Tel_t=Tel[:]
        iter_1=0
    
        while len(index)>0 and iter_1<3:
            #shift PPCs with 0.2 deg, but only run three iterations to save computational time
            #typically one iteration is enough 
            for ind in index:
                Tel_t[ind,1]=Tel[ind,1]+np.random.choice([-.2,.2],1)[0]
                Tel_t[ind,2]=Tel[ind,2]+np.random.choice([-.2,.2],1)[0]
            
            res,telescope,tgt=netflowRun_single(Tel_t,sample)
            index=np.where(np.array([len(tt) for tt in res])==0)[0]
            
            iter_1+=1
        
        return res,telescope,tgt

def netflowRun(sample,Print=False):
    '''run netflow (with iteration and DBSCAN)
        
    Parameters
    ==========
    sample : sample
        
    Returns
    =======
    Fiber assignment in each PPC
    '''
    time_start=time.time()
    
    if Print: print("#########Step 3/4: Start running netflow to assign fibers")
        
    ppc_g=point_DBSCAN(sample,False,Print) #separate ppc into different groups
    
    point_list=[]
    point_c=0
    
    for uu in range(len(ppc_g)): #run netflow for each ppc group
        #only consider sample in the group
        sample_index=list(chain.from_iterable([list(PFS_FoV(ppc_g[uu][iii,1],ppc_g[uu][iii,2],ppc_g[uu][iii,3],sample)) \
                                               for iii in range(len(ppc_g[uu]))]))
        if len(sample_index)==0:
            continue
        sample_inuse=sample[list(set(sample_index))]
            
        res,telescope,tgt=netflowRun_nofibAssign(ppc_g[uu],sample_inuse,Print)
        
        for i, (vis,  tel) in enumerate(zip(res,  telescope)):
            fib_eff_t=len(vis)/2394.*100      
            
            #assigned targets in each ppc
            obj_allo_id=[]
            for tidx, cidx in vis.items():
                obj_allo_id.append(tgt[tidx].ID)  
            
            #calculate the total weights in each ppc (smaller value means more important)
            if len(vis)==0:
                tot_weight=np.nan
            else:
                tot_weight=1/sum(sample[np.in1d(sample['obj_id'],obj_allo_id)]['weight'])
                
            point_list.append(["Point_"+str(point_c+1),"Group_"+str(uu+1),tel._ra, tel._dec, \
                               tel._posang,tot_weight,fib_eff_t,obj_allo_id,sample['resolution'][0]])
            point_c+=1
    
    point_t=Table(np.array(point_list,dtype=object),names=['point_id','group_id','tel_ra','tel_dec',\
                                                           'tel_pa','tel_priority','tel_fiber_usage_frac',\
                                                           'allocated_targets','resolution'],\
                 dtype=[np.str_,np.str_,np.float64,np.float64,np.float64,np.float64,np.float64,object,np.str_])
    
    if Print:
        print("#########Step 3/4: Run netflow DONE! (takes",round(time.time()-time_start,3),"sec)")
        
    return point_t

def complete_ppc(sample,point_l):
    '''examine the completeness fraction of the user sample

    Parameters
    ==========
    sample : table
    
    point_l: table of ppc information
        
    Returns
    =======
    sample with allocated time
    
    completion rate: in each user-defined priority + overall
    '''
    sample.add_column(0,name='allocate_time')
    point_l_pri=point_l[point_l.argsort(keys='tel_priority')] #sort ppc by its total priority == sum(weights of the assigned targets in ppc)
    
    #sub-groups of the input sample, catagarized by the user defined priority
    sub_l=sorted(list(set(sample['priority'])))
    n_sub=len(sub_l)
    count_sub=[sum(sample['priority']==ll) for ll in sub_l]+[len(sample)]
    completeR=[] # count
    completeR_=[] # percentage
    
    for ppc in point_l_pri:
        lst=np.where(np.in1d(sample['obj_id'],ppc['allocated_targets'])==True)[0]
        sample['allocate_time'].data[lst]+=900
        
        comp_s= np.where(sample['exptime_PPP']==sample['allocate_time'])[0]
        comT_t=[sum(sample['priority'].data[comp_s]==ll) for ll in sub_l]
        comT_t.append(len(comp_s))
        completeR.append(comT_t)  
        completeR_.append([comT_t[oo]/count_sub[oo]*100 for oo in range(len(count_sub))])
        
    return sample,np.array(completeR),np.array(completeR_),sub_l

def plotCR(cR,sub,obj_allo):     
    
    plt.figure(figsize=(20,6))

    plt.subplot(121)
    plt.plot(np.arange(1,len(cR)+1,1),cR[:,-1],'k-',lw=4,zorder=10)
    
    n_sub=len(sub)
    colors = sns.color_palette(cc.glasbey_warm, n_sub)
    for uu in range(n_sub):
        if sub[uu]==min(sub):
            plt.plot(np.arange(1,len(cR)+1,1),cR[:,uu],'-',color=colors[uu],lw=2,zorder=5,\
                     label='P_'+str(sub[uu]))
        else:
            plt.plot(np.arange(1,len(cR)+1,1),cR[:,uu],'--',color=colors[uu],lw=1,zorder=3,\
                     label='P_'+str(sub[uu]))
    
    #forbidden area, obs_time>5 nights
    plt.axvspan(4*8*5,len(cR)+1,fc='k',alpha=0.3,zorder=10)
    
    #GradeA area, need to be given by observatory ahead
    area1=plt.Rectangle((30,88), 95, 12, fc='orange',ec="none",alpha=0.3)
    plt.gca().add_patch(area1)
    
    #GradeB area, need to be given by observatory ahead
    area2=plt.Rectangle((20,43), 130, 50, fc='dodgerblue',ec="none",alpha=0.3)
    plt.gca().add_patch(area2)
        
    plt.xlim(0,len(cR)+1)
    plt.ylim(0,cR.max()+1)
    plt.xlabel('PPC',fontsize=18)
    plt.ylabel('completeness',fontsize=18)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend(loc='best',fontsize=12)
    plt.grid()

    plt.subplot(122)
    
    obj_allo=obj_allo[obj_allo.argsort(keys='tel_priority')]
    fib_eff=obj_allo['tel_fiber_usage_frac'].data
    
    plt.bar(np.arange(0,len(fib_eff),1),fib_eff,width=0.8,fc='tomato',ec='none',alpha=0.6,zorder=10)
    plt.plot([0,len(fib_eff)],[80,80],'k--',lw=2,zorder=11)
    plt.plot([0,len(fib_eff)],[np.mean(fib_eff),np.mean(fib_eff)],'--',color='tomato',lw=2,zorder=11)
    plt.text(len(fib_eff)*0.8,np.mean(fib_eff),"{:2.2f}%".format(np.mean(fib_eff)),color='tomato',\
             fontsize=12)
    
    plt.xlim(0,len(fib_eff)+1)
    plt.ylim(0,max(fib_eff)+1.)
    plt.xlabel('PPC',fontsize=18)
    plt.ylabel('fiber alloc fraction',fontsize=18)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.grid()
    plt.show()
    
def overheads(n_sci_frame):
    #in seconds
    t_exp_sci: float = 900.
    t_exp_bias: float = 0.
    t_exp_dark: float = 900.
    t_exp_flat: float = 10.
    t_exp_arc: float = 10.
    t_lamp_flat: float = 60.
    t_lamp_arc: float = 60.
    t_focus: float = 300.
    t_overhead_misc: float = 60.
    t_overhead_fiber: float = 180.
        
    n_frame_bias: float = 10.
    n_frame_dark: float = 10.
    n_frame_flat: float = 10.
    n_frame_arc: float = 10.
    n_focus: float = 3.
    
    #t_tot_bias = (t_exp_bias + t_overhead_misc) * n_frame_bias
    #t_tot_dark = (t_exp_dark + t_overhead_misc) * n_frame_dark
    
    #total time for calibration
    t_calib=t_lamp_flat+(t_exp_flat+t_overhead_misc) * n_frame_flat + \
            t_lamp_arc+(t_exp_arc+t_overhead_misc) * n_frame_arc 
    t_focus_tot=(t_focus*n_focus)
    
    #maximal number of pointings for each night
    n_sci_perNight=(10*3600-t_calib-t_focus_tot)/(t_exp_sci+t_overhead_misc+t_overhead_fiber)
    
    #the required number of night
    n_sci_night=np.ceil(n_sci_frame/n_sci_perNight)
    
    #total required time
    Toverheads_tot_best = (t_calib+t_focus_tot)*n_sci_night+(t_exp_sci+t_overhead_misc+t_overhead_fiber)*n_sci_frame
    
    Toverheads_tot_worst = (t_overhead_fiber+t_calib+t_exp_sci+t_overhead_misc+t_overhead_fiber)*n_sci_frame+\
                           t_focus_tot*n_sci_night 
    
    return Toverheads_tot_best/3600.,Toverheads_tot_worst/3600.
    
def printCR(RESmode,obj_allo):
    hour_tot=len(obj_allo)*15./60. # hour
    Fhour_tot=sum([len(tt) for tt in obj_allo['allocated_targets']])*15./60. # fiber_count*hour
    Ttot_best, Ttot_worst=overheads(len(obj_allo))
    
    print(
    f'''
    This is for the {RESmode:s}-resolution mode.
    #------------------------------------------------------------------------------
    The total number of pointings you required: {len(obj_allo):4d}
    The total on-source time you required: {hour_tot:.2f} hours; {Fhour_tot:.2f} fiber hours
    The total time (inlcuding overheads) you required: {Ttot_best:.2f} hours (best case); {Ttot_worst:.2f} hours (worst case)''')
 
    if hour_tot>8*5: # required time > 5 nights (assuming 8 hours per night, 1 hour=4 ppc)
        print('  !!!WARNING: the total on-source time you required exceeds the upper limit (5 nights).')
        
    fib_eff_mean=np.mean(obj_allo['tel_fiber_usage_frac'].data)
    fib_eff_small=sum(obj_allo['tel_fiber_usage_frac']<30)/len(obj_allo)*100.
    
    print(
    f'''
    The average fiber usage fraction: {fib_eff_mean:.2f} %
    There are {fib_eff_small:.2f}% of the total pointings having fiber usage fraction less than 30%.''')

def netflow_iter(uS,obj_allo,conta,contb,contc,PrintTF=True):
    '''iterate the total procedure to re-assign fibers to targets which have not been assigned 
        in the previous/first iteration 

    Parameters
    ==========
    uS: table
        sample with exptime>allocate_time
        
    obj_allo: table
        ppc information
        
    conta,contb,contc: float
        weight parameters
        
    PrintTF: boolean , default==True
        
    Returns
    =======
    table of ppc information after all targets are assigned
    # note that some targets in the dense region may need very long time to be assigned with fibers
        # if targets can not be successfully assigned with fibers in >5 iterations, then directly stop
        # if total number of ppc >200 (~5 nights), then directly stop
    '''
    time_start=time.time()
    if sum(uS['allocate_time']==uS['exptime_PPP'])==len(uS):
        #remove ppc with no fiber assignment
        obj_allo.remove_rows(np.where(obj_allo['tel_fiber_usage_frac']==0)[0])
        if PrintTF:
            print("#########Step 4/4: Iterate DONE! (takes",round(time.time()-time_start,3),"sec)")
        return obj_allo
    
    else:       
        #  select non-assigned targets --> PPC determination --> netflow --> if no fibre assigned: shift PPC
        iter_m2=0

        while any(uS['allocate_time']<uS['exptime_PPP']) and iter_m2<10:
            uS_t1=uS[uS['allocate_time']<uS['exptime_PPP']]
            uS_t1['exptime_PPP']=uS_t1['exptime_PPP']-uS_t1['allocate_time'] #remained exposure time
            uS_t1.remove_column('allocate_time')
        
            uS_t2=PPP_centers(uS_t1,True,conta,contb,contc,False)
            
            obj_allo_t=netflowRun(uS_t2,False)
                            
            if len(obj_allo)>200 or iter_m2>=10:
                #stop if n_ppc>200
                return obj_allo
            
            else:
                obj_allo =vstack([obj_allo, obj_allo_t]) 
                obj_allo.remove_rows(np.where(obj_allo['tel_fiber_usage_frac']==0)[0])
                uS=complete_ppc(uS_t2,obj_allo)[0]                
                iter_m2+=1
                
        if PrintTF:
            print("#########Step 4/4: Iterate DONE! (takes",round(time.time()-time_start,3),"sec)")
                
        return obj_allo

In [56]:
thread=20
uS,uS_L,uS_M=s1
conta,contb,contc=[4.02,0.22,0.15]

_ = mr.Note(text="###### Step3. Fiber assignment")
Netrun = mr.Button(label="Start",style='success')
if Netrun.clicked:
    if len(uS_L)>0: #for low-resolution mode    
        obj_allo_L =netflowRun(uS_L_s2,True)

    if len(uS_M)>0: #for medium-resolution mode
        obj_allo_M =netflowRun(uS_M_s2,True)

###### Step3. Fiber assignment

mercury.Button

In [55]:
thread=20
uS,uS_L,uS_M=s1
conta,contb,contc=[4.02,0.22,0.15]

_ = mr.Note(text="###### Step4. Iterate")
Itrun = mr.Button(label="Start",style='success')
if Itrun.clicked:
    if len(uS_L)>0: #for low-resolution mode    
        uS_L2=complete_ppc(uS_L_s2,obj_allo_L)[0]
        obj_allo_L_fin=netflow_iter(uS_L2,obj_allo_L,conta,contb,contc,True)
        
        uS_L_s2.remove_column('allocate_time')
        uS_L2,cR_L,cR_L_,sub_l=complete_ppc(uS_L_s2,obj_allo_L_fin)

    if len(uS_M)>0: #for medium-resolution mode
        
        uS_M2=complete_ppc(uS_M_s2,obj_allo_M)[0]
        obj_allo_M_fin=netflow_iter(uS_M2,obj_allo_M,conta,contb,contc,True)
        
        uS_M_s2.remove_column('allocate_time')
        
        uS_M2,cR_M,cR_M_,sub_m=complete_ppc(uS_M_s2,obj_allo_M_fin)
        

###### Step4. Iterate

mercury.Button

In [6]:
_ = mr.Note(text="###### Please see results:")
Morun = mr.Checkbox(value=False, label="show results of the low-resolution mode")
if Morun.value:
    if len(uS_L)>0:
        _ = mr.Note(text="You can modify the number of PPC:")
        n_ppc_l=len(obj_allo_L_fin)
        PPC_slider_l = mr.Slider(value=n_ppc_l, min=0, max=n_ppc_l, label=" ", step=1)
        
        n_ppc_user_l=PPC_slider_l.value
        
        printCR('low',obj_allo_L_fin[:n_ppc_user_l])  
        plotCR(cR_L_[:n_ppc_user_l],sub_l,obj_allo_L_fin[:n_ppc_user_l])
        
    else:
        _ = mr.Note(text="You do not have targets requiring the low-resolution mode.")

###### Please see results:

mercury.Checkbox

In [4]:
Morun = mr.Checkbox(value=False, label="show results of the medium-resolution mode")
if Morun.value:
    if len(uS_M)>0:
        _ = mr.Note(text="You can modify the number of PPC:")
        n_ppc_m=len(obj_allo_M_fin)
        PPC_slider_m = mr.Slider(value=n_ppc_m, min=0, max=n_ppc_m, label=" ", step=1)
        
        n_ppc_user_m=PPC_slider_m.value
        
        printCR('medium',obj_allo_M_fin[:n_ppc_user_m])  
        plotCR(cR_M_[:n_ppc_user_m],sub_m,obj_allo_M_fin[:n_ppc_user_m])
        
    else:
        _ = mr.Note(text="You do not have targets requiring the medium-resolution mode.")

mercury.Checkbox

In [5]:
Morun = mr.Checkbox(value=False, label="show results of the L&M-resolution mode")
if Morun.value:
    if len(uS_L)>0 and len(uS_M)>0:  
        printCR('L&M',vstack([obj_allo_L_fin[:n_ppc_user_l],obj_allo_M_fin[:n_ppc_user_m]]))

mercury.Checkbox